# 12.3 · Transformer 从零实现 / Transformer from Scratch ⭐⭐⭐

> **课程定位 / Where this fits**
> 第 3 课，**Part 12**。**全 Part 12 的核心**，也是理解 BERT/GPT/所有 LLM 的钥匙。
> Lesson 3, **Part 12**. **The heart of Part 12** and the key to understanding BERT/GPT and every LLM.
>
> 2017 年 *"Attention Is All You Need"* 提出 **Transformer**：**彻底抛弃 RNN 的循环**，只用**自注意力 + 前馈网络**堆叠而成。两大杀手锏：①**任意距离依赖**——任意两个位置一步直连(不像 RNN 要逐步传递)；②**完全并行**——所有位置同时计算(RNN 必须串行)，训练快几个数量级。这让"大力出奇迹"的大模型成为可能。本课**从零实现** Transformer 的每个零件：**缩放点积注意力、多头注意力、位置编码、残差+LayerNorm**，并搭出完整模型训练+可视化。
> In 2017 *"Attention Is All You Need"* introduced the **Transformer**: **drop RNN recurrence entirely**, stacking only **self-attention + feedforward**. Two superpowers: ① **any-distance dependencies** — any two positions connect in one step (no step-by-step passing); ② **full parallelism** — all positions computed at once (RNNs are serial), training orders of magnitude faster. This enabled today's massive LLMs. We **implement every piece from scratch**: **scaled dot-product attention, multi-head attention, positional encoding, residual + LayerNorm**, building a full model with training + visualization.
>
> 💼 **实战/面试视角**：Transformer 是 LLM 面试的**绝对核心**——"自注意力计算 / 为什么除以√d / 多头的作用 / 位置编码为什么需要 / 残差+LN" 几乎必考。
> 💼 **Practical/interview angle:** the Transformer is the **absolute core** of LLM interviews — self-attention math, the √d scaling, multi-head purpose, why positional encoding, residual+LN — almost guaranteed.

> 📐 **符号约定 / Notation**
> - $Q,K,V$ —— query/key/value 矩阵 / query/key/value
> - $d_k$ —— 每个头的维度 / per-head dimension
> - $\text{Attn}=\text{softmax}(QK^\top/\sqrt{d_k})V$ / scaled dot-product attention

> 💡 **面试相关 / Interview-relevant**
> - "自注意力的完整计算"（出镜率 ★★★★★）
> - "为什么要除以 √d_k"（★★★★★，防softmax饱和)
> - "多头注意力的作用"（★★★★★)
> - "位置编码为什么必需、有哪些"（★★★★★)
> - "残差连接+LayerNorm 的作用 / Pre-LN vs Post-LN"（★★★★)
> - "Transformer 相比 RNN 的优势"（★★★★★，并行+长依赖)

---

## 学习目标 / Learning Objectives
1. 理解 Transformer 为何取代 RNN(并行 + 任意距离)。
   Understand why Transformers replaced RNNs (parallel + any-distance).
2. **从零实现缩放点积注意力**，懂 √d 缩放。
   Implement scaled dot-product attention from scratch; understand √d scaling.
3. **从零实现多头注意力**，懂多头的意义。
   Implement multi-head attention; understand its purpose.
4. 实现并**可视化位置编码**。
   Implement and visualize positional encoding.
5. 组装 **Transformer 块**(MHA+FFN+残差+LN)训练并可视化注意力。
   Assemble Transformer blocks (MHA+FFN+residual+LN), train, visualize attention.

## 目录 / TOC
1. [从 RNN 到 Transformer ⭐](#1)
2. [缩放点积注意力（从零）⭐](#2)
3. [多头注意力（从零）⭐](#3)
4. [位置编码 ⭐](#4)
5. [组装 Transformer 块 + 训练 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 从 RNN 到 Transformer ⭐ / From RNN to Transformer

RNN(12.1)和注意力(12.2)各有贡献，但 RNN 仍有两大硬伤：
RNNs (12.1) and attention (12.2) each contributed, but RNNs still have two hard limits:
- **必须串行**：第 $t$ 步要等第 $t-1$ 步算完(隐藏状态依赖)。无法并行 → 训练慢、用不满 GPU。
  **Serial:** step $t$ waits for step $t-1$ (hidden-state dependency). No parallelism → slow, underuses GPUs.
- **长依赖仍吃力**：信息要逐步传递，越远越难。
  **Long deps still hard:** info passes step by step, harder over distance.

Transformer 的洞见：**既然注意力能让任意两个位置直接交互，那干脆把 RNN 整个扔掉，只用注意力！** 这样：
The Transformer insight: **since attention lets any two positions interact directly, drop the RNN entirely and use only attention!** Then:
- **任意距离 = 一步**：位置 1 和位置 1000 直接算注意力，无需穿过 999 步。
  **Any distance = one hop:** position 1 and 1000 attend directly, no 999-step relay.
- **完全并行**：所有位置的注意力**同时**算(就是几个大矩阵乘法)，GPU 满载。这是大模型能训练的**根本原因**。
  **Fully parallel:** all positions' attention computed **at once** (big matmuls), saturating GPUs — the **fundamental reason** large models are trainable.

代价：注意力**不知道顺序**(它对输入是个集合)，所以要额外加**位置编码**(§4)；且复杂度是 $O(n^2)$(每对位置都算)。下面从零搭出每个零件。
The cost: attention is **order-agnostic** (treats input as a set), so we add **positional encoding** (§4); and complexity is $O(n^2)$ (every pair). Let's build each piece from scratch.


<a id="2"></a>
## 2. 缩放点积注意力（从零）⭐ / Scaled Dot-Product Attention

回顾(10.9/11.3)：每个 token 生成 **Query/Key/Value**。注意力 = 用 $Q$ 和所有 $K$ 算相似度 → softmax → 对 $V$ 加权求和。完整公式：
Recall (10.9/11.3): each token produces **Query/Key/Value**. Attention = score $Q$ against all $K$ → softmax → weighted sum of $V$. The full formula:

$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

**关键: 为什么除以 $\sqrt{d_k}$**(面试高频)：$Q\cdot K$ 是 $d_k$ 个数相乘相加，维度越高，点积的方差越大、数值越大。若不缩放，过大的分数会让 **softmax 进入饱和区**(输出接近 one-hot)，**梯度几乎为 0**，训练不动。除以 $\sqrt{d_k}$ 把方差拉回 1，保持梯度健康。
**Key: why divide by $\sqrt{d_k}$** (high-frequency): $Q\cdot K$ sums $d_k$ products; higher dim → larger variance/magnitude. Without scaling, large scores push **softmax into saturation** (near one-hot) where **gradients ≈ 0** and training stalls. Dividing by $\sqrt{d_k}$ restores variance ≈ 1, keeping gradients healthy.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, math
import torch, torch.nn as nn, torch.nn.functional as F
sns.set_theme(style="whitegrid")
torch.manual_seed(0); np.random.seed(0)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """从零的缩放点积注意力 / scaled dot-product attention from scratch.
       Q,K,V: (..., seq, d_k)."""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)     # 相似度并缩放 √d_k / scaled scores
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))   # 掩码位置→-inf(softmax后≈0) / mask
    attn = F.softmax(scores, dim=-1)                      # 注意力权重(每行和为1) / attention weights
    return attn @ V, attn                                 # 加权求和 V / weighted sum of V

# 演示 √d 缩放的重要性: 高维下不缩放会让 softmax 饱和 / show why scaling matters
for d in [8, 128]:
    q = torch.randn(1, d); k = torch.randn(6, d)
    raw = (q @ k.T).squeeze()                              # 未缩放分数 / unscaled
    scaled = raw / math.sqrt(d)
    print(f"d={d:3}: 未缩放分数范围 [{raw.min():.1f},{raw.max():.1f}], softmax最大权重={F.softmax(raw,0).max():.3f}")
    print(f"       缩放后  分数范围 [{scaled.min():.1f},{scaled.max():.1f}], softmax最大权重={F.softmax(scaled,0).max():.3f}")
print("\n高维(d=128)不缩放 → 分数很大 → softmax 几乎 one-hot(饱和) → 梯度消失; 除以√d 缓解")


<a id="3"></a>
## 3. 多头注意力（从零）⭐ / Multi-Head Attention

**单个注意力**只能学一种"关注模式"。**多头注意力(multi-head)**：把 $d$ 维拆成 $h$ 个**头**，每个头在自己的 $d/h$ 维子空间里独立做注意力，最后把各头结果**拼接**再线性变换。
A **single attention** learns one "focus pattern." **Multi-head attention** splits the $d$ dims into $h$ **heads**, each doing attention independently in its $d/h$-dim subspace, then **concatenates** the heads and projects.

**为什么多头**(面试核心)：不同头可以**关注不同类型的关系**——一个头看"主谓一致"、一个看"指代"、一个看"邻近词"……就像 CNN 用多个卷积核学不同特征(10.2)。多头让模型在同一层捕捉多种依赖。
**Why multi-head** (interview core): different heads attend to **different kinds of relations** — one for subject-verb agreement, one for coreference, one for adjacency… like CNN's multiple kernels learning different features (10.2). Multi-head captures diverse dependencies in one layer.


In [ ]:
class MultiHeadAttention(nn.Module):
    """从零的多头自注意力 / multi-head self-attention from scratch."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h = n_heads; self.d_k = d_model // n_heads   # 每个头的维度 / per-head dim
        self.qkv = nn.Linear(d_model, 3*d_model)          # 一次性投影出 Q,K,V / project to Q,K,V at once
        self.out = nn.Linear(d_model, d_model)            # 输出投影 / output projection
    def forward(self, x, mask=None):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.h, self.d_k).permute(2, 0, 3, 1, 4)  # 拆成3×头 / split
        Q, K, V = qkv[0], qkv[1], qkv[2]                  # 各 (B, h, T, d_k) / each head
        out, attn = scaled_dot_product_attention(Q, K, V, mask)   # 每个头独立注意力 / per-head attention
        out = out.transpose(1, 2).reshape(B, T, D)        # 拼接各头 / concatenate heads
        return self.out(out), attn                        # 输出投影 / output projection

mha = MultiHeadAttention(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)                                # (batch, seq, d_model)
out, attn = mha(x)
print(f"多头注意力: 输入 {tuple(x.shape)} → 输出 {tuple(out.shape)}")
print(f"注意力权重形状 {tuple(attn.shape)} = (batch, 头数=4, seq=10, seq=10)")
print("4个头并行, 每个在 64/4=16 维子空间做注意力, 各学不同关注模式, 再拼接")


<a id="4"></a>
## 4. 位置编码 ⭐ / Positional Encoding

自注意力**对顺序无感**：打乱输入词的顺序，每个词的注意力输出不变(它只看"和谁相似"，不看"谁在前")。但语言里**顺序至关重要**("狗咬人"≠"人咬狗")。所以必须**显式注入位置信息**——**位置编码(positional encoding)**：给每个位置一个向量，加到词向量上。
Self-attention is **order-agnostic**: shuffle the input words and each word's attention output is unchanged (it only sees "who's similar," not "who's first"). But order is **crucial** in language ("dog bites man" ≠ "man bites dog"). So we must **inject position explicitly** — **positional encoding**: a vector per position, added to the word embeddings.

两种主流(面试)：
Two main kinds (interview):
- **正弦位置编码(sinusoidal)**：用不同频率的 sin/cos 函数算出固定向量(原论文)。优点：可外推到比训练更长的序列。
  **Sinusoidal:** fixed vectors from sin/cos at varying frequencies (original paper). Pro: extrapolates to longer-than-trained sequences.
- **可学习位置编码(learned)**：直接学一个"位置→向量"的查表(BERT/GPT 用)。本课模型用它。
  **Learned:** a learned position→vector table (used by BERT/GPT). Our model uses this.

下面实现并**可视化正弦位置编码**——它独特的条纹图案让每个位置有独一无二的"指纹"。
Below we implement and **visualize sinusoidal positional encoding** — its stripe pattern gives each position a unique "fingerprint."


In [ ]:
def sinusoidal_pe(max_len, d_model):
    """正弦位置编码 PE(pos,2i)=sin(pos/10000^(2i/d)), PE(pos,2i+1)=cos(...) / sinusoidal PE."""
    pe = np.zeros((max_len, d_model))
    pos = np.arange(max_len)[:, None]
    div = np.exp(np.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))  # 不同维度不同频率 / varying freq
    pe[:, 0::2] = np.sin(pos * div)                       # 偶数维用 sin / even dims: sin
    pe[:, 1::2] = np.cos(pos * div)                       # 奇数维用 cos / odd dims: cos
    return pe

PE = sinusoidal_pe(60, 64)
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(PE, cmap="RdBu_r", center=0, cbar_kws={"label":"编码值"}, ax=ax)
ax.set_xlabel("编码维度"); ax.set_ylabel("序列位置"); ax.set_title("正弦位置编码: 每个位置一行独特'指纹'(不同频率的sin/cos条纹)")
plt.tight_layout(); plt.show()
print("自注意力无序→必须加位置编码; 正弦版用多频率sin/cos给每个位置唯一编码, 可外推长序列")
print("现代 GPT/BERT 多用'可学习位置编码'(直接学位置→向量表); 还有 RoPE 等相对位置编码")


<a id="5"></a>
## 5. 组装 Transformer 块 + 训练 + 小结 ⭐ / Assembling Blocks, Training & Summary

一个 **Transformer 块** = **多头自注意力** + **前馈网络(FFN)**，每个子层都裹上**残差连接 + LayerNorm**：
A **Transformer block** = **multi-head self-attention** + a **feedforward network (FFN)**, each sublayer wrapped with **residual connection + LayerNorm**:
- **残差连接**(呼应 10.4)：`x + sublayer(x)`，给梯度高速公路，让深层 Transformer 可训练。
  **Residual** (echoing 10.4): `x + sublayer(x)`, a gradient highway making deep Transformers trainable.
- **LayerNorm**：对每个 token 的特征做归一化，稳定训练(注意：NLP 用 LayerNorm 而非 BatchNorm，因为序列长度可变、批内样本不该相互影响)。
  **LayerNorm:** normalize each token's features, stabilizing training (NLP uses LayerNorm not BatchNorm: variable lengths, samples shouldn't affect each other).
- **FFN**：两层全连接(中间放大4倍+激活)，给每个位置做非线性变换。
  **FFN:** two linear layers (4× wider middle + activation), a per-position nonlinear transform.

> **Pre-LN vs Post-LN**(面试)：原论文把 LN 放在子层**之后**(Post-LN)；现代实践把 LN 放**之前**(Pre-LN)，训练更稳、更易收敛。我们用 Pre-LN。
> **Pre-LN vs Post-LN:** the original placed LN **after** sublayers (Post-LN); modern practice puts it **before** (Pre-LN) for more stable training. We use Pre-LN.

我们把若干块堆起来，在 **12.2 的反转任务**上训练(反转长度相同，用 encoder-only 即可)，看 Transformer 在 L=16 上轻松达到完美——而 12.2 的无注意力 RNN 在 L=12 就已崩溃。
We stack blocks and train on **12.2's reverse task** (same length → encoder-only suffices). The Transformer nails L=16 easily — where 12.2's no-attention RNN already collapsed at L=12.


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=256):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
    def forward(self, x, mask=None):
        h, attn = self.attn(self.norm1(x), mask)          # Pre-LN: 先归一化再注意力 / pre-LN
        x = x + h                                         # 残差 / residual
        x = x + self.ffn(self.norm2(x))                   # 残差 FFN / residual FFN
        return x, attn

class TinyTransformer(nn.Module):
    def __init__(self, vocab, d_model=64, n_heads=4, n_layers=2, max_len=20):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = nn.Parameter(torch.randn(1, max_len, d_model)*0.02)   # 可学习位置编码 / learned PE
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads) for _ in range(n_layers)])
        self.head = nn.Linear(d_model, vocab)
    def forward(self, x):
        h = self.emb(x) + self.pos[:, :x.size(1)]         # 词向量 + 位置编码 / token + position
        attns = []
        for blk in self.blocks: h, a = blk(h); attns.append(a)
        return self.head(h), attns

# 反转任务数据 / reverse-task data
V = 12; DIG = list(range(2, V))
def gen(n, L):
    s = [list(np.random.choice(DIG, L)) for _ in range(n)]; return torch.tensor(s), torch.tensor([x[::-1] for x in s])
Xtr, Ytr = gen(4000, 16); Xte, Yte = gen(800, 16)
torch.manual_seed(0); model = TinyTransformer(V); opt = torch.optim.Adam(model.parameters(), 3e-3); ce = nn.CrossEntropyLoss()
for ep in range(15):
    for i in range(0, len(Xtr), 128):
        lg, _ = model(Xtr[i:i+128]); opt.zero_grad(); ce(lg.reshape(-1,V), Ytr[i:i+128].reshape(-1)).backward(); opt.step()
lg, attns = model(Xte); acc = (lg.argmax(2)==Yte).all(1).float().mean().item()
print(f"从零搭的 Transformer 在反转任务(L=16)上整句准确率 = {acc:.3f}")
print(f"参数量 {sum(p.numel() for p in model.parameters()):,}; 对比: 12.2 无注意力 RNN 在 L=12 就已崩溃")

# 可视化多头注意力(应呈反对角线: 位置i关注位置L-1-i) / visualize heads (anti-diagonal)
A = attns[-1][0].detach().numpy()                         # 最后一层, 第0个样本, 各头 / last layer, sample 0
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for hi, ax in enumerate(axes):
    sns.heatmap(A[hi], cmap="viridis", cbar=False, ax=ax); ax.set_title(f"头 {hi}", fontsize=10)
    ax.set_xlabel("被关注位置"); ax.set_ylabel("查询位置" if hi==0 else "")
fig.suptitle("多头自注意力(反转任务): 反对角线 = 每个位置关注其镜像位置(模型自己学到)")
plt.tight_layout(); plt.show()
print("注意力呈反对角线 → Transformer 学会'位置i该看位置L-1-i'; 多个头模式可不同")


```
Transformer: 抛弃RNN循环, 只用自注意力+FFN堆叠; ①任意距离一步直连 ②完全并行(训练快)→ 大模型可行
缩放点积注意力: softmax(QK^T/√d_k)V; 除√d_k 防点积过大→softmax饱和→梯度消失
多头注意力: d拆成h个头各做注意力再拼接; 不同头学不同关系(像CNN多核); 一层捕捉多种依赖
位置编码: 自注意力无序→必须加位置; 正弦(可外推)/可学习(BERT/GPT)/RoPE(相对位置)
Transformer块: 多头注意力 + FFN, 每个子层 残差(梯度高速公路)+LayerNorm(NLP用LN非BN); Pre-LN更稳
复杂度 O(n²): 每对位置都算注意力 → 长序列贵(后续有各种高效注意力)
三种用法: 编码器(双向,BERT) / 解码器(因果掩码,GPT) / 编码-解码(翻译,T5)
```

### 💡 面试速查 / Interview cheat-sheet
1. **自注意力**: softmax(QK^T/√d_k)V; 任意位置直接交互。
   Self-attention: softmax(QK^T/√d_k)V; any positions interact directly.
2. **√d_k 缩放**: 防点积过大使softmax饱和→梯度消失。
   √d_k scaling: prevents large dot products saturating softmax → vanishing gradients.
3. **多头**: 拆成多个子空间各学不同关注模式再拼接。
   Multi-head: split into subspaces, learn different patterns, concatenate.
4. **位置编码**: 注意力无序故必加; 正弦/可学习/相对(RoPE)。
   Positional encoding: required since attention is order-free; sinusoidal/learned/relative.
5. **残差+LN+并行**: 残差=梯度高速公路, LN稳训练(非BN), 全并行=训练快(取代RNN关键)。
   Residual+LN+parallel: residual highway, LayerNorm (not BN), full parallelism (key over RNN).

### 下一节 / Next
**12.4 BERT 与编码器模型**——Transformer 的**编码器**怎么用？BERT 用**双向**自注意力 + **掩码语言模型(MLM)** 预训练，成为理解类任务(分类/抽取/检索)的霸主。我们会从零实现 MLM 预训练的核心思想。
**12.4 BERT & Encoder Models** — how to use the Transformer **encoder**? BERT uses **bidirectional** self-attention + **masked language modeling (MLM)** pretraining, dominating understanding tasks (classification/extraction/retrieval). We'll implement the core idea of MLM pretraining from scratch.
